# Build a Purchasing Star and Link It to Sales

**Course:** Data Pipelines with Databricks and Fabric  
**Dataset:** Microsoft AdventureWorks — Purchasing  
**Catalog:** `adventure_works`

**Medallion pipeline:** Bronze (raw staging) → Silver (named, typed, cleaned) → Gold (`fact_purchases`)

**Result:** Two stars sharing `dim_product`:
- **Sales star (earlier lessons):** `default.fact_sales` + `default.dim_product` + `default.dim_sales_territory`
- **Purchasing star (this notebook):** `silver.fact_purchases` + `silver.dim_vendor` (+ shared `default.dim_product`)

> **Before running:** the 3 raw purchasing CSVs (`PurchaseOrderDetail.csv`, `PurchaseOrderHeader.csv`,
> `Vendor.csv`) must be in `/Volumes/adventure_works/default/raw_data/` (tab-delimited, no header row).
> Run all cells top to bottom.

In [0]:
# Inspect the raw CSV files before loading: delimiter, column count, first data row
# The earlier Lesson 1 CSVs were tab-delimited with no header row — verify the purchasing CSVs match
RAW_DATA = "/Volumes/adventure_works/default/raw_data"

for csv_file in ["PurchaseOrderDetail.csv", "PurchaseOrderHeader.csv", "Vendor.csv"]:
    path = f"{RAW_DATA}/{csv_file}"
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().rstrip("\n\r")
    tab_count = first_line.count("\t")
    comma_count = first_line.count(",")
    delimiter = "TAB" if tab_count > comma_count else "COMMA"
    print(f"=== {csv_file} ===")
    print(f"  delimiter: {delimiter} | columns: {max(tab_count, comma_count) + 1}")
    print(f"  first row preview: {first_line[:120]}\n")

In [0]:
# Look up the real column names and types in the CREATE TABLE statements of instawdb.sql
# (Microsoft's SQL Server install script — the single source of truth for the schema)

with open(f"{RAW_DATA}/instawdb.sql", "r", encoding="utf-8", errors="ignore") as f:
    ddl_lines = f.readlines()

for table_name in ["PurchaseOrderDetail", "PurchaseOrderHeader", "Vendor"]:
    marker = f"CREATE TABLE [Purchasing].[{table_name}]"
    for i, line in enumerate(ddl_lines):
        if marker in line:
            print(f"===== {table_name} (instawdb.sql line {i + 1}) =====")
            for l in ddl_lines[i + 1:]:
                s = l.strip()
                if s.startswith("CONSTRAINT") or s.startswith(")"):
                    break  # column list ends where constraints / the closing paren begin
                if s:
                    print(f"  {s}")
            print()
            break

## Bronze Layer — Raw Staging Tables

**What:** load the 3 raw CSVs exactly as they are, as 3 separate staging tables in `adventure_works.bronze`.

**Why:** Bronze captures raw data untouched. The CSVs have **no header row**, so reading with `header=False` makes Spark assign generic column names — `_c0`, `_c1`, `_c2`, … — exactly what the exercise expects. The real names are applied in Silver.

**Pattern:** `read CSV (tab-delimited) → saveAsTable (adventure_works.bronze)`

In [0]:
# BRONZE — load the 3 raw CSVs as staging tables
# header=False  -> columns come in as _c0, _c1, _c2 ... (CSVs have no header row)
# sep='\t'      -> the CSV extracts are tab-delimited
# No schema/inference -> every column stays STRING: truly raw, untouched data

bronze_sources = {
    "PurchaseOrderDetail.csv": "stg_purchase_order_detail",
    "PurchaseOrderHeader.csv": "stg_purchase_order_header",
    "Vendor.csv": "dim_vendor",
}

for csv_file, table_name in bronze_sources.items():
    df = (
        spark.read.format("csv")
        .option("header", False)
        .option("sep", "\t")
        .load(f"{RAW_DATA}/{csv_file}")
    )
    df.write.mode("overwrite").saveAsTable(f"adventure_works.bronze.{table_name}")
    print(f"{csv_file} -> adventure_works.bronze.{table_name}: {df.count()} rows")

In [0]:
# BRONZE verification — confirm the generic _c0, _c1 ... column names and row counts
for table_name in bronze_sources.values():
    print(f"===== adventure_works.bronze.{table_name} =====")
    display(spark.sql(f"DESCRIBE adventure_works.bronze.{table_name}"))

## Silver Layer — Real Names, Real Types, Data Quality

**What:** take the 3 Bronze staging tables, apply the real column names from `instawdb.sql`, cast the SQL Server types to Databricks types, and enforce a data-quality rule.

**Type mapping (SQL Server → Databricks):**  
| SQL Server | Databricks |
| --- | --- |
| int | INT |
| smallint / tinyint | SMALLINT |
| money | DECIMAL(19,4) |
| decimal(8,2) | DECIMAL(8,2) |
| datetime | TIMESTAMP |
| nvarchar / Name / AccountNumber | STRING |
| Flag | BOOLEAN |

**Data quality:** on `stg_purchase_order_detail`, drop rows with a NULL `ProductID` or a negative `LineTotal`.

In [0]:
# SILVER 1/2 — stg_purchase_order_detail: real column names + types + data-quality filter
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.silver")

# Column list and types straight from CREATE TABLE [Purchasing].[PurchaseOrderDetail]
# (instawdb.sql line 1454): _c0->PurchaseOrderID, _c1->PurchaseOrderDetailID, ...
detail_columns = [
    ("PurchaseOrderID", "int"),
    ("PurchaseOrderDetailID", "int"),
    ("DueDate", "timestamp"),
    ("OrderQty", "smallint"),
    ("ProductID", "int"),
    ("UnitPrice", "decimal(19,4)"),      # money
    ("LineTotal", "decimal(19,4)"),       # computed in SQL Server, present in the CSV
    ("ReceivedQty", "decimal(8,2)"),
    ("RejectedQty", "decimal(8,2)"),
    ("StockedQty", "decimal(8,2)"),        # computed in SQL Server, present in the CSV
    ("ModifiedDate", "timestamp"),
]

bronze_detail = spark.table("adventure_works.bronze.stg_purchase_order_detail")

# Apply the real names + types, then enforce the data-quality rule:
# drop rows with a NULL ProductID or a negative LineTotal
silver_detail = bronze_detail.select(
    *[F.col(f"_c{i}").cast(t).alias(c) for i, (c, t) in enumerate(detail_columns)]
).filter(
    F.col("ProductID").isNotNull() & (F.col("LineTotal") >= 0)
)

silver_detail.write.mode("overwrite").saveAsTable("adventure_works.silver.stg_purchase_order_detail")

before = bronze_detail.count()
after = spark.table("adventure_works.silver.stg_purchase_order_detail").count()
print(f"stg_purchase_order_detail: {before} -> {after} rows (dropped {before - after}: NULL ProductID or negative LineTotal)")

In [0]:
# SILVER 2/2 — stg_purchase_order_header and dim_vendor: real column names + types
from pyspark.sql import functions as F

# From CREATE TABLE [Purchasing].[PurchaseOrderHeader] (instawdb.sql line 1473)
header_columns = [
    ("PurchaseOrderID", "int"),
    ("RevisionNumber", "smallint"),      # tinyint
    ("Status", "smallint"),             # tinyint; 1=Pending, 2=Approved, 3=Rejected, 4=Complete
    ("EmployeeID", "int"),
    ("VendorID", "int"),
    ("ShipMethodID", "int"),
    ("OrderDate", "timestamp"),
    ("ShipDate", "timestamp"),           # nullable
    ("SubTotal", "decimal(19,4)"),       # money
    ("TaxAmt", "decimal(19,4)"),
    ("Freight", "decimal(19,4)"),
    ("TotalDue", "decimal(19,4)"),       # computed in SQL Server, present in the CSV
    ("ModifiedDate", "timestamp"),
]

# From CREATE TABLE [Purchasing].[Vendor] (instawdb.sql line 1756)
vendor_columns = [
    ("BusinessEntityID", "int"),
    ("AccountNumber", "string"),
    ("Name", "string"),
    ("CreditRating", "smallint"),          # tinyint
    ("PreferredVendorStatus", "boolean"),  # Flag
    ("ActiveFlag", "boolean"),             # Flag
    ("PurchasingWebServiceURL", "string"),
    ("ModifiedDate", "timestamp"),
]

for bronze_table, silver_table, columns in [
    ("stg_purchase_order_header", "stg_purchase_order_header", header_columns),
    ("dim_vendor", "dim_vendor", vendor_columns),
]:
    (
        spark.table(f"adventure_works.bronze.{bronze_table}")
        .select(*[F.col(f"_c{i}").cast(t).alias(c) for i, (c, t) in enumerate(columns)])
        .write.mode("overwrite")
        .saveAsTable(f"adventure_works.silver.{silver_table}")
    )
    rows = spark.table(f"adventure_works.silver.{silver_table}").count()
    print(f"adventure_works.silver.{silver_table}: {rows} rows, {len(columns)} columns")

In [0]:
# SILVER verification — real names and real types, no _c0 columns left
for table_name in ["stg_purchase_order_detail", "stg_purchase_order_header", "dim_vendor"]:
    print(f"===== adventure_works.silver.{table_name} =====")
    display(spark.sql(f"DESCRIBE adventure_works.silver.{table_name}"))

## Gold Layer — fact_purchases

**What:** join `stg_purchase_order_detail` to `stg_purchase_order_header` on `PurchaseOrderID` to pull in `VendorID` and `OrderDate`, one row per PO line. Save as the single fact table `fact_purchases`, next to the Silver `dim_vendor`.

**Why:** the detail rows know *what* was bought (product, qty, price) but not *who from* or *when it was ordered* — that lives in the header. The join completes the purchasing star:

- **Purchasing star:** `fact_purchases` + `dim_vendor` (sharing `dim_product` with the sales star)
- **Sales star:** `fact_sales` + `dim_product` + `dim_sales_territory`

In [0]:
# GOLD — build fact_purchases: detail + header joined on PurchaseOrderID
# Pre-check: PurchaseOrderID must be unique in the header — otherwise the join would fan out
dup_po_ids = spark.sql("""
    SELECT COUNT(*) AS duplicate_po_ids
    FROM (
        SELECT PurchaseOrderID
        FROM adventure_works.silver.stg_purchase_order_header
        GROUP BY PurchaseOrderID
        HAVING COUNT(*) > 1
    )
""").collect()[0]["duplicate_po_ids"]
print(f"Duplicate PurchaseOrderIDs in header: {dup_po_ids} (0 = safe to join, no fan-out)")

spark.sql("""
CREATE OR REPLACE TABLE adventure_works.silver.fact_purchases AS
SELECT
    d.PurchaseOrderID,
    d.PurchaseOrderDetailID,
    d.DueDate,
    d.OrderQty,
    d.ProductID,
    d.UnitPrice,
    d.LineTotal,
    d.ReceivedQty,
    d.RejectedQty,
    d.StockedQty,
    h.VendorID,        -- pulled in from the header
    h.OrderDate,       -- pulled in from the header
    d.ModifiedDate
FROM adventure_works.silver.stg_purchase_order_detail d
JOIN adventure_works.silver.stg_purchase_order_header h
  ON d.PurchaseOrderID = h.PurchaseOrderID
""")

display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM adventure_works.silver.stg_purchase_order_detail) AS detail_rows,
    (SELECT COUNT(*) FROM adventure_works.silver.stg_purchase_order_header) AS header_rows,
    (SELECT COUNT(*) FROM adventure_works.silver.fact_purchases) AS fact_rows
"""))
# detail_rows = fact_rows proves the join kept one row per PO line (no fan-out)

## Query 1 — Thinnest Margin Between Sales Price and Purchase Price

Join `fact_sales` and `fact_purchases` on `ProductID` (both share `dim_product`).

A product can be sold and purchased at different prices over time, so compare the **average** sales price against the **average** purchase price per product. The thinnest margin = smallest difference (ordering ascending can even surface products sold below cost).

In [0]:
%sql
-- Top 5 products with the thinnest margin between sales price and purchase price
-- Average each price per product FIRST, then join: joining the raw facts directly
-- on ProductID would be many-to-many and distort the averages.
-- Zero prices (unsellable raw-material components) are excluded.
WITH sales_avg AS (
    SELECT ProductID, AVG(UnitPrice) AS avg_sales_price
    FROM adventure_works.default.fact_sales
    WHERE UnitPrice > 0
    GROUP BY ProductID
),
purchase_avg AS (
    SELECT ProductID, AVG(UnitPrice) AS avg_purchase_price
    FROM adventure_works.silver.fact_purchases
    WHERE UnitPrice > 0
    GROUP BY ProductID
)
SELECT
    p.Name AS product_name,
    ROUND(s.avg_sales_price, 4) AS avg_sales_price,
    ROUND(pu.avg_purchase_price, 4) AS avg_purchase_price,
    ROUND(s.avg_sales_price - pu.avg_purchase_price, 4) AS avg_margin
FROM sales_avg s
JOIN purchase_avg pu ON s.ProductID = pu.ProductID
JOIN adventure_works.default.dim_product p ON s.ProductID = p.ProductID
ORDER BY avg_margin ASC
LIMIT 5;

## Query 2 — Units Sold vs. Units Purchased, by Month

Which products sell faster than we restock them?  
Aggregate units sold (`fact_sales`) and units purchased (`fact_purchases`) per product per month, join on both, and keep only months where sold > purchased (a restocking shortfall).

*Note: the sales month uses `ModifiedDate` — the true order date lives in `SalesOrderHeader`, which is not loaded in this lakehouse. Purchase month uses `OrderDate` from the header.*

In [0]:
%sql
-- Products that sell faster than we restock them, by month
-- FULL OUTER JOIN keeps the complete monthly comparison: months with sales only,
-- purchases only, or both. The WHERE keeps months where units_sold > units_purchased.
WITH sold AS (
    SELECT ProductID,
           date_trunc('month', ModifiedDate) AS month,
           SUM(OrderQty) AS units_sold
    FROM adventure_works.default.fact_sales
    GROUP BY 1, 2
),
purchased AS (
    SELECT ProductID,
           date_trunc('month', OrderDate) AS month,
           SUM(OrderQty) AS units_purchased
    FROM adventure_works.silver.fact_purchases
    GROUP BY 1, 2
)
SELECT
    p.Name AS product_name,
    COALESCE(s.month, pu.month) AS month,
    COALESCE(s.units_sold, 0) AS units_sold,
    COALESCE(pu.units_purchased, 0) AS units_purchased,
    COALESCE(s.units_sold, 0) - COALESCE(pu.units_purchased, 0) AS units_shortfall
FROM sold s
FULL OUTER JOIN purchased pu
  ON s.ProductID = pu.ProductID AND s.month = pu.month
JOIN adventure_works.default.dim_product p
  ON COALESCE(s.ProductID, pu.ProductID) = p.ProductID
WHERE COALESCE(s.units_sold, 0) > COALESCE(pu.units_purchased, 0)
ORDER BY units_shortfall DESC, units_sold DESC
LIMIT 20;

## Result — One Lakehouse, Two Stars Sharing dim_product

| Star | Fact | Dimensions |
| --- | --- | --- |
| **Sales** | `default.fact_sales` (121,317 rows) | `default.dim_product`, `default.dim_sales_territory` |
| **Purchasing** | `silver.fact_purchases` | `silver.dim_vendor` (+ shared `default.dim_product`) |

Medallion flow: **Bronze** raw staging (`_c0` names) → **Silver** named/typed/cleaned → **Gold** `fact_purchases` joined one-row-per-PO-line.